# Iteration 1.5 – fixed BiGRU + Naive Bayes

| | Iteration 1.5 |
|---|---|
| **Model** | the BiGRU of iteration 1, simplified, + Naive Bayes hints |
| **Libraries** | `torch` (PyTorch), `numpy`, `pandas` (Naive Bayes written by hand in numpy) |
| **How text becomes numbers** | words → word vectors learned from scratch → GRU (max 512 words); Naive Bayes scores words and word pairs and gives the network one hint per label |
| **Input** | description + rank + category + cantrip + casting time |
| **Output** | traits, traditions, save, area, damage types, sustained |
| **Data** | same 1,920 spells and split as iteration 1 |
| **Result** | micro-F1 **0.740** |
| **Test: only rank + description** | micro-F1 **0.647** |
| **Status** | ⚠️ much better than iteration 1, still behind iteration 2 |

## What changed compared to iteration 1

Iteration 1.5 keeps the **model family and the libraries of iteration 1**: a PyTorch network with word embeddings learned from scratch and a bidirectional GRU, written with `numpy`, `pandas` and `torch` only. It is simplified and receives the fixes that made iteration 2 work. Naive Bayes is added to the network as an extra input.

| | Iteration 1 | Iteration 1.5 |
|---|---|---|
| Text cleaning | Foundry links deleted, cut at 256 words | links rendered as words (`@Damage[2d6[fire]]` → `2d6 fire`), cut at 512 words (16 of 1,920 spells are longer) |
| Input | text + `rank / 10` | text + rank, category, cantrip and casting time as one-hot input |
| Naive Bayes | – | per-label Naive Bayes log-odds as an extra input to the network |
| Targets | 3,705 keywords + 100 traits | traits, traditions, save, area, damage types, sustained (same as iteration 2) |
| Network | embedding → BiGRU → 2-layer classifier → 2 heads | embedding → BiGRU → 1 hidden layer → 1 head per target group |
| Padding | padding read by the backward GRU | packed sequences, padding is skipped |
| Class imbalance | `pos_weight` up to 20 | `pos_weight` = √ratio, capped at 10 |
| Training | 15 epochs, best checkpoint by loss | early stopping (patience 5) + learning-rate decay |
| Threshold | fixed 0.35 | tuned on the validation set per group |

The test spells are the same 192 spells as in iterations 1, 2 and 3, so the results can be compared directly.  (AI geenratel tabs)

In [1]:
from __future__ import annotations

import copy
import html
import json
import random
import re
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

ROOT = Path.cwd()
DATA_DIR = ROOT.parents[1] / "data" / "spellsdata"
ARTIFACT_DIR = ROOT / "artifacts"
SEED = 42
MAX_TEXT_LENGTH = 512
MIN_TOKEN_FREQUENCY = 2
MIN_LABEL_SUPPORT = 5
NB_FOLDS = 5
TRADITIONS = ["arcane", "divine", "occult", "primal"]
CLASS_TRAITS = {
    "animist", "bard", "champion", "cleric", "druid", "exemplar", "kineticist", "magus", "monk",
    "necromancer", "oracle", "psychic", "ranger", "sorcerer", "summoner", "thaumaturge", "witch", "wizard",
}
RULE_TRAITS = {"cantrip", "focus"}
FULL_INPUT = ["category", "is_cantrip", "rank", "cast_time"]
ORIGINAL_IDEA = ["rank"]  # the original idea: the user only types the rank and the description

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print(f"data_dir={DATA_DIR}")
print(f"config: max_tokens={MAX_TEXT_LENGTH}, min_token_frequency={MIN_TOKEN_FREQUENCY}, nb_folds={NB_FOLDS}")

data_dir=/Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/data/spellsdata
config: max_tokens=512, min_token_frequency=2, nb_folds=5


## Stage 1: text cleaning

The same cleaning as in iteration 2. Foundry inline commands are rendered as the text a player would see instead of being deleted, and numbers and dice are kept.

In [2]:
INLINE_PATTERN = re.compile(r"@(UUID|Damage|Check|Template|Embed|Localize)\[")
HTML_PATTERN = re.compile(r"<[^>]+>")
RANDOM_ID_PATTERN = re.compile(r"(?=.*\d)[A-Za-z0-9]{16}")
TOKEN_PATTERN = re.compile(r"[a-z0-9][a-z0-9'+-]*")

def closing_bracket(value: str, start: int) -> int:
    depth = 0
    for index in range(start, len(value)):
        if value[index] == "[":
            depth += 1
        elif value[index] == "]":
            depth -= 1
            if depth == 0:
                return index
    return len(value) - 1

def render_inline(kind: str, body: str, label: str | None) -> str:
    if label:
        return label
    if kind == "UUID":
        name = body.rsplit(".", 1)[-1]
        return " " if RANDOM_ID_PATTERN.fullmatch(name) else name
    if kind == "Damage":
        return " ".join(re.sub(r"[\[\],]", " ", body).split())
    if kind == "Check":
        parts = body.split("|")
        is_basic = any(part.split(":")[0] == "basic" for part in parts[1:])
        return f"basic {parts[0]} save" if is_basic else f"{parts[0]} check"
    if kind == "Template":
        params = dict(part.split(":", 1) for part in body.split("|") if ":" in part)
        shape = params.get("type", body.split("|")[0])
        return f"{params['distance']}-foot {shape}" if "distance" in params else shape
    return " "

def expand_inline(value: str) -> str:
    parts: list[str] = []
    position = 0
    while (match := INLINE_PATTERN.search(value, position)):
        body_end = closing_bracket(value, match.end() - 1)
        body = value[match.end():body_end]
        end, label = body_end, None
        if body_end + 1 < len(value) and value[body_end + 1] == "{":
            label_end = value.find("}", body_end + 1)
            end = len(value) - 1 if label_end == -1 else label_end
            label = value[body_end + 2:end]
        parts.append(value[position:match.start()])
        parts.append(f" {render_inline(match.group(1), body, label)} ")
        position = end + 1
    parts.append(value[position:])
    return "".join(parts)

def clean_text(value: str) -> str:
    value = expand_inline(value or "")
    value = HTML_PATTERN.sub(" ", value)
    return " ".join(html.unescape(value).lower().split())

def tokenize(value: str) -> list[str]:
    return TOKEN_PATTERN.findall(value)

example_raw = ("Creatures are knocked @UUID[Compendium.pf2e.conditionitems.Item.Prone] and take "
               "@Damage[2d6[persistent,fire]] damage (@Check[reflex|dc:20|basic]).")
print("raw:    ", example_raw)
print("tokens: ", tokenize(clean_text(example_raw)))

raw:     Creatures are knocked @UUID[Compendium.pf2e.conditionitems.Item.Prone] and take @Damage[2d6[persistent,fire]] damage (@Check[reflex|dc:20|basic]).
tokens:  ['creatures', 'are', 'knocked', 'prone', 'and', 'take', '2d6', 'persistent', 'fire', 'damage', 'basic', 'reflex', 'save']


## Stage 2: dataset, structured input and targets

The same rows, structured input and target groups as iteration 2:

* **input:** cleaned text + category (spell / focus / ritual / impossible), cantrip yes/no, rank and casting time;
* **targets:** traits, traditions, damage types and sustained (multi-label), save and area (one choice each);
* `cantrip` and `focus` follow from the input by a rule; class traits and traits with fewer than 5 spells are not predicted.

In [3]:
def spell_category(path: Path) -> str:
    top_folder = path.relative_to(DATA_DIR).parts[0]
    return {"spells": "spell", "focus": "focus", "rituals": "ritual", "impossible-spells": "impossible"}[top_folder]

def cast_time_bucket(value: Any) -> str:
    text = str(value or "").strip().lower()
    if text in {"1", "2", "3", "reaction", "free"}:
        return text
    if re.fullmatch(r"\d (to|or) \d.*", text):
        return "variable"
    if "minute" in text or "hour" in text:
        return "exploration"
    if "day" in text or "week" in text:
        return "downtime"
    return "other"

def area_bucket(value: str | None) -> str:
    if not value:
        return "none"
    return value if value in {"burst", "emanation", "cone", "line"} else "other"

def load_spells(data_dir: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for path in sorted(data_dir.rglob("*.json")):
        if path.name == "_folders.json":
            continue
        payload = json.loads(path.read_text(encoding="utf-8"))
        system = payload.get("system", {})
        traits_data = system.get("traits", {})
        raw_text = system.get("description", {}).get("value", "")
        traits = sorted({str(label).casefold() for label in traits_data.get("value", []) if label})
        traditions = sorted({str(label).casefold() for label in traits_data.get("traditions", []) if label})
        if not raw_text.strip() or not (traits or traditions):
            continue
        defense = system.get("defense") or {}
        text = clean_text(raw_text)
        rows.append({
            "name": payload.get("name", path.stem),
            "text": text,
            "tokens": tokenize(text),
            "category": spell_category(path),
            "is_cantrip": "cantrip" in traits,
            "rank": int(system.get("level", {}).get("value", 0) or 0),
            "cast_time": cast_time_bucket(system.get("time", {}).get("value")),
            "traits": traits,
            "traditions": traditions,
            "save": (defense.get("save") or {}).get("statistic") or "none",
            "area": area_bucket((system.get("area") or {}).get("type")),
            "damage_types": sorted({entry.get("type") for entry in (system.get("damage") or {}).values() if entry.get("type")}),
            "sustained": bool((system.get("duration") or {}).get("sustained")),
        })
    return pd.DataFrame(rows)

def rule_traits(category: str, is_cantrip: bool) -> list[str]:
    labels = ["cantrip"] if is_cantrip else []
    if category == "focus" and not is_cantrip:
        labels.append("focus")
    return labels

print("\n[1/5] Loading spells, structured input and targets...")
df = load_spells(DATA_DIR)
trait_support = Counter(trait for labels in df["traits"] for trait in labels)
trait_labels = sorted(trait for trait, count in trait_support.items()
                      if count >= MIN_LABEL_SUPPORT and trait not in CLASS_TRAITS | RULE_TRAITS)
damage_support = Counter(label for labels in df["damage_types"] for label in labels)
damage_labels = sorted(label for label, count in damage_support.items() if count >= MIN_LABEL_SUPPORT)
MULTILABEL_GROUPS = {"traits": trait_labels, "traditions": TRADITIONS, "damage_types": damage_labels, "sustained": ["sustained"]}
MULTICLASS_GROUPS = {"save": ["none", "fortitude", "reflex", "will"],
                     "area": ["none", "burst", "emanation", "cone", "line", "other"]}
ALL_GROUPS = [*MULTILABEL_GROUPS, *MULTICLASS_GROUPS]

def group_targets(frame: pd.DataFrame, group: str) -> np.ndarray:
    if group in MULTICLASS_GROUPS:
        return frame[group].map(MULTICLASS_GROUPS[group].index).to_numpy()
    labels = MULTILABEL_GROUPS[group]
    column = frame[group].map(lambda value: ["sustained"] if value is True else [] if value is False else value)
    return np.array([[label in row for label in labels] for row in column], dtype=np.int64)

print(f"  usable spells={len(df)}")
print(f"  traits={len(trait_labels)}, traditions={len(TRADITIONS)}, damage types={len(damage_labels)}, save=4 classes, area=6 classes")
print(f"  token length: median={int(df['tokens'].map(len).median())}, longer than {MAX_TEXT_LENGTH}: {int((df['tokens'].map(len) > MAX_TEXT_LENGTH).sum())} spells")


[1/5] Loading spells, structured input and targets...


  usable spells=1920
  traits=66, traditions=4, damage types=16, save=4 classes, area=6 classes
  token length: median=113, longer than 512: 16 spells


## Stage 3: split, vocabulary and Naive Bayes input

**Split:** the same `sample(frac=1, random_state=42)` 80/10/10 split as every other iteration.

**Vocabulary:** words that occur at least twice in the training split. Iteration 1 kept every word, which gave 8,702 embeddings to learn from 1,536 spells.

**Naive Bayes input.** For every label, Naive Bayes measures how much more often a word or word pair occurs in spells with that label than in spells without it (log-count ratio $r$). The sum of $r$ over the words of a spell plus the label prior is the Naive Bayes log-odds: a first guess per label. These guesses (one number per label, 97 in total) are given to the network next to the GRU output, and the GRU learns when to correct them.

If Naive Bayes were fitted on the same spells it scores, its guesses for the training spells would be too good and the network would learn to trust them blindly. So the scores of the training spells are computed **out-of-fold**: Naive Bayes is fitted on 4/5 of the training split and scores the remaining 1/5. Validation and test spells are scored by Naive Bayes fitted on the whole training split.

In [4]:
def split_frame(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    shuffled = frame.sample(frac=1, random_state=SEED).reset_index(drop=True)
    test_start = int(len(shuffled) * 0.8)
    valid_start = int(len(shuffled) * 0.9)
    return shuffled.iloc[:test_start], shuffled.iloc[test_start:valid_start], shuffled.iloc[valid_start:]

train_df, valid_df, test_df = split_frame(df)
frames = {"train": train_df, "valid": valid_df, "test": test_df}
Y = {split: {group: group_targets(frame, group) for group in ALL_GROUPS} for split, frame in frames.items()}

token_frequency = Counter(token for tokens in train_df["tokens"] for token in tokens)
vocabulary = [token for token, count in token_frequency.most_common() if count >= MIN_TOKEN_FREQUENCY]
token_to_id = {"<PAD>": 0, "<UNK>": 1, **{token: index + 2 for index, token in enumerate(vocabulary)}}

structured_values = {column: sorted(train_df[column].astype(str).unique()) for column in FULL_INPUT}

def structured_matrix(frame: pd.DataFrame, columns: list[str]) -> np.ndarray:
    blocks = [np.array([[str(value) == option for option in structured_values[column]] for value in frame[column]], dtype=np.float32)
              for column in columns]
    return np.hstack(blocks)

def ngrams(tokens: list[str]) -> set[str]:
    return set(tokens) | {f"{first} {second}" for first, second in zip(tokens, tokens[1:])}

ngram_frequency = Counter(gram for tokens in train_df["tokens"] for gram in ngrams(tokens))
ngram_to_id = {gram: index for index, gram in enumerate(sorted(gram for gram, count in ngram_frequency.items() if count >= 2))}

def presence_matrix(frame: pd.DataFrame) -> np.ndarray:
    matrix = np.zeros((len(frame), len(ngram_to_id)), dtype=np.float32)
    for row, tokens in enumerate(frame["tokens"]):
        matrix[row, [ngram_to_id[gram] for gram in ngrams(tokens) if gram in ngram_to_id]] = 1.0
    return matrix

def nb_targets(split: str) -> np.ndarray:
    blocks = []
    for group in ALL_GROUPS:
        targets = Y[split][group]
        blocks.append(np.eye(len(MULTICLASS_GROUPS[group]))[targets] if group in MULTICLASS_GROUPS else targets)
    return np.hstack(blocks).astype(np.float32)

def nb_fit(X: np.ndarray, targets: np.ndarray, alpha: float = 1.0) -> tuple[np.ndarray, np.ndarray]:
    p = alpha + targets.T @ X
    q = alpha + (1 - targets).T @ X
    ratio = np.log(p / p.sum(axis=1, keepdims=True)) - np.log(q / q.sum(axis=1, keepdims=True))
    positives = targets.sum(axis=0)
    prior = np.log((positives + alpha) / (len(targets) - positives + alpha))
    return ratio, prior

def nb_log_odds(X: np.ndarray, ratio: np.ndarray, prior: np.ndarray) -> np.ndarray:
    return X @ ratio.T + prior

print("\n[2/5] Split, vocabulary and Naive Bayes input...")
X_presence = {split: presence_matrix(frame) for split, frame in frames.items()}
nb_train_targets = nb_targets("train")
out_of_fold = np.zeros((len(train_df), nb_train_targets.shape[1]), dtype=np.float32)
fold_of_row = np.random.default_rng(SEED).permutation(len(train_df)) % NB_FOLDS
for fold in range(NB_FOLDS):
    held_out = fold_of_row == fold
    ratio, prior = nb_fit(X_presence["train"][~held_out], nb_train_targets[~held_out])
    out_of_fold[held_out] = nb_log_odds(X_presence["train"][held_out], ratio, prior)
nb_ratio, nb_prior = nb_fit(X_presence["train"], nb_train_targets)
nb_scores = {"train": out_of_fold,
             "valid": nb_log_odds(X_presence["valid"], nb_ratio, nb_prior),
             "test": nb_log_odds(X_presence["test"], nb_ratio, nb_prior)}
nb_mean, nb_std = nb_scores["train"].mean(axis=0), nb_scores["train"].std(axis=0) + 1e-6

print(f"  split sizes: train={len(train_df)}, valid={len(valid_df)}, test={len(test_df)}")
print(f"  GRU vocabulary={len(token_to_id):,} (iteration 1: 8,702), Naive Bayes n-grams={len(ngram_to_id):,}")
print(f"  structured one-hot={structured_matrix(train_df, FULL_INPUT).shape[1]}, Naive Bayes log-odds per spell={nb_train_targets.shape[1]}")
feature_names = sorted(ngram_to_id, key=ngram_to_id.get)
for trait in ["fire", "illusion", "teleportation", "mental"]:
    row = nb_ratio[trait_labels.index(trait)]
    print(f"  NB words for {trait:13s} -> {', '.join(feature_names[i] for i in np.argsort(row)[::-1][:6])}")


[2/5] Split, vocabulary and Naive Bayes input...


  split sizes: train=1536, valid=192, test=192
  GRU vocabulary=5,352 (iteration 1: 8,702), Naive Bayes n-grams=22,118
  structured one-hot=24, Naive Bayes log-odds per spell=97
  NB words for fire          -> 3d6 fire, 1d4 fire, the flame, a fiery, a roaring, ignition
  NB words for illusion      -> disbelieve, to disbelieve, disbelieve the, the illusory, disbelieves, disbelieves the
  NB words for teleportation -> you teleport, extradimensional container, teleport the, same planet, transport yourself, transposition
  NB words for mental        -> target's mind, becomes frightened, creature's mind, persistent mental, distracted, actions against


## Stage 4: simplified BiGRU

* **Embedding (96) → bidirectional GRU (96 per direction)**, as in iteration 1, but with packed sequences, so the backward direction starts at the last real word instead of at the padding.
* The GRU output (mean over the real words) is joined with the structured one-hot input and, in the Naive Bayes variant, the standardized Naive Bayes log-odds.
* **Naive Bayes shortcut:** in the Naive Bayes variant each output starts from that label's Naive Bayes guess (times a learned scale), and the network only learns a correction on top of it. Without this shortcut the GRU overfits in a few epochs and ignores the Naive Bayes input.
* **One hidden layer (128) with dropout, then one head per target group.** The keyword head and the second classifier layer of iteration 1 are gone.
* The loss is averaged per group, so no group dominates. Multi-label groups use BCE with `pos_weight` = √(negatives / positives), capped at 10. `save` and `area` use cross-entropy.

In [5]:
def encode_tokens(tokens: list[str]) -> list[int]:
    ids = [token_to_id.get(token, token_to_id["<UNK>"]) for token in tokens[:MAX_TEXT_LENGTH]]
    return ids or [token_to_id["<UNK>"]]

def build_inputs(split: str, use_naive_bayes: bool, columns: list[str]) -> dict[str, Tensor]:
    encoded = [encode_tokens(tokens) for tokens in frames[split]["tokens"]]
    input_ids = torch.zeros((len(encoded), MAX_TEXT_LENGTH), dtype=torch.long)
    for row, ids in enumerate(encoded):
        input_ids[row, :len(ids)] = torch.tensor(ids)
    extra = structured_matrix(frames[split], columns)
    if use_naive_bayes:
        extra = np.hstack([extra, (nb_scores[split] - nb_mean) / nb_std])
    return {"input_ids": input_ids, "lengths": torch.tensor([len(ids) for ids in encoded]),
            "extra": torch.tensor(extra, dtype=torch.float32)}

def build_targets(split: str) -> dict[str, Tensor]:
    return {group: torch.tensor(Y[split][group], dtype=torch.float32 if group in MULTILABEL_GROUPS else torch.long)
            for group in ALL_GROUPS}

NB_SLICES, start = {}, 0
for group in ALL_GROUPS:
    size = len(MULTICLASS_GROUPS[group]) if group in MULTICLASS_GROUPS else len(MULTILABEL_GROUPS[group])
    NB_SLICES[group] = (start, start + size)
    start += size
NB_DIM = start  # the Naive Bayes guesses are the last NB_DIM columns of the extra input

class SpellTaggerV15(nn.Module):
    def __init__(self, vocab_size: int, extra_dim: int, output_sizes: dict[str, int], use_naive_bayes: bool,
                 embedding_dim: int = 96, hidden_dim: int = 96, dropout: float = 0.3) -> None:
        super().__init__()
        self.use_naive_bayes = use_naive_bayes
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.encoder = nn.GRU(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.shared = nn.Sequential(nn.Linear(hidden_dim * 2 + extra_dim, 128), nn.ReLU(), nn.Dropout(dropout))
        self.text_dropout = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({group: nn.Linear(128, size) for group, size in output_sizes.items()})
        # Naive Bayes shortcut: every label starts from its own Naive Bayes guess, the GRU learns a correction.
        self.nb_scale = nn.ParameterDict({group: nn.Parameter(torch.ones(size)) for group, size in output_sizes.items()})

    def forward(self, input_ids: Tensor, lengths: Tensor, extra: Tensor) -> dict[str, Tensor]:
        packed = pack_padded_sequence(self.embedding(input_ids), lengths, batch_first=True, enforce_sorted=False)
        encoded, _ = pad_packed_sequence(self.encoder(packed)[0], batch_first=True, total_length=input_ids.size(1))
        mask = (torch.arange(input_ids.size(1)).unsqueeze(0) < lengths.unsqueeze(1)).unsqueeze(-1)
        pooled = (encoded * mask).sum(dim=1) / lengths.unsqueeze(1)
        shared = self.shared(torch.cat([self.text_dropout(pooled), extra], dim=1))
        outputs = {group: head(shared) for group, head in self.heads.items()}
        if self.use_naive_bayes:
            naive_bayes = extra[:, -NB_DIM:]
            for group, (first, last) in NB_SLICES.items():
                outputs[group] = outputs[group] + self.nb_scale[group] * naive_bayes[:, first:last]
        return outputs

POS_WEIGHTS = {group: torch.tensor(np.sqrt((len(train_df) - counts) / np.maximum(counts, 1)).clip(1, 10), dtype=torch.float32)
               for group, counts in ((group, Y["train"][group].sum(axis=0)) for group in MULTILABEL_GROUPS)}
OUTPUT_SIZES = {**{group: len(labels) for group, labels in MULTILABEL_GROUPS.items()},
                **{group: len(labels) for group, labels in MULTICLASS_GROUPS.items()}}

def multitask_loss(outputs: dict[str, Tensor], targets: dict[str, Tensor]) -> Tensor:
    loss = torch.zeros(())
    for group, logits in outputs.items():
        if group in MULTILABEL_GROUPS:
            loss = loss + nn.functional.binary_cross_entropy_with_logits(logits, targets[group], pos_weight=POS_WEIGHTS[group])
        else:
            loss = loss + nn.functional.cross_entropy(logits, targets[group])
    return loss

print("\n[3/5] Creating the simplified BiGRU...")
example_model = SpellTaggerV15(len(token_to_id), build_inputs("valid", True, FULL_INPUT)["extra"].shape[1], OUTPUT_SIZES, use_naive_bayes=True)
print("  architecture: Embedding -> packed BiGRU -> [GRU output + structured input + Naive Bayes log-odds] -> hidden layer -> 6 heads")
print("  each head output = Naive Bayes guess x learned scale + GRU correction")
print(f"  parameters={sum(parameter.numel() for parameter in example_model.parameters()):,} (iteration 1: 1,479,325)")


[3/5] Creating the simplified BiGRU...
  architecture: Embedding -> packed BiGRU -> [GRU output + structured input + Naive Bayes log-odds] -> hidden layer -> 6 heads
  each head output = Naive Bayes guess x learned scale + GRU correction
  parameters=678,338 (iteration 1: 1,479,325)


## Stage 5: training, evaluation and ablation

The network is trained twice with the same settings, once without and once with the Naive Bayes input, so the effect of Naive Bayes is visible separately from the other fixes. Thresholds are tuned on the validation set per group. The scores use the same definitions as iterations 2 and 3:

* micro-F1;
* macro-F1 over all labels and over the labels present in the test set;
* the comparable score in the 100-label space of iteration 1.

In [6]:
def batches(size: int, batch_size: int, shuffle: bool) -> list[Tensor]:
    order = torch.randperm(size) if shuffle else torch.arange(size)
    return [order[start:start + batch_size] for start in range(0, size, batch_size)]

def run_model(model: SpellTaggerV15, inputs: dict[str, Tensor], index: Tensor) -> dict[str, Tensor]:
    width = int(inputs["lengths"][index].max())
    return model(inputs["input_ids"][index, :width], inputs["lengths"][index], inputs["extra"][index])

@torch.no_grad()
def predict(model: SpellTaggerV15, inputs: dict[str, Tensor]) -> dict[str, np.ndarray]:
    model.eval()
    outputs: dict[str, list[np.ndarray]] = {group: [] for group in ALL_GROUPS}
    for index in batches(len(inputs["lengths"]), 128, shuffle=False):
        for group, logits in run_model(model, inputs, index).items():
            probabilities = logits.sigmoid() if group in MULTILABEL_GROUPS else logits.softmax(dim=1)
            outputs[group].append(probabilities.numpy())
    return {group: np.concatenate(values) for group, values in outputs.items()}

@torch.no_grad()
def dataset_loss(model: SpellTaggerV15, inputs: dict[str, Tensor], targets: dict[str, Tensor]) -> float:
    model.eval()
    total = 0.0
    for index in batches(len(inputs["lengths"]), 128, shuffle=False):
        total += multitask_loss(run_model(model, inputs, index), {group: value[index] for group, value in targets.items()}).item() * len(index)
    return total / len(inputs["lengths"])

def train(use_naive_bayes: bool, columns: list[str] = FULL_INPUT, max_epochs: int = 40,
          patience: int = 5) -> tuple[SpellTaggerV15, list[dict[str, float]]]:
    set_seed()
    inputs = {split: build_inputs(split, use_naive_bayes, columns) for split in ["train", "valid"]}
    targets = {split: build_targets(split) for split in ["train", "valid"]}
    model = SpellTaggerV15(len(token_to_id), inputs["train"]["extra"].shape[1], OUTPUT_SIZES, use_naive_bayes)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
    best_loss, best_state, waited, history = float("inf"), None, 0, []
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for index in batches(len(train_df), 32, shuffle=True):
            loss = multitask_loss(run_model(model, inputs["train"], index), {group: value[index] for group, value in targets["train"].items()})
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(index)
        valid_loss = dataset_loss(model, inputs["valid"], targets["valid"])
        scheduler.step(valid_loss)
        improved = valid_loss < best_loss - 1e-4
        history.append({"epoch": epoch, "train_loss": train_loss / len(train_df), "valid_loss": valid_loss})
        print(f"  epoch={epoch:02d} train_loss={train_loss / len(train_df):.4f} valid_loss={valid_loss:.4f}{' * best' if improved else ''}")
        if improved:
            best_loss, best_state, waited = valid_loss, copy.deepcopy(model.state_dict()), 0
        else:
            waited += 1
            if waited >= patience:
                print(f"  early stopping: no improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)
    return model, history

THRESHOLD_GRID = np.round(np.concatenate([[0.02, 0.03], np.arange(0.05, 0.96, 0.05)]), 2)

def f1_scores(predicted: np.ndarray, targets: np.ndarray) -> dict[str, float]:
    predicted, targets = predicted.astype(bool), targets.astype(bool)
    tp = (predicted & targets).sum(axis=0)
    fp = (predicted & ~targets).sum(axis=0)
    fn = (~predicted & targets).sum(axis=0)
    per_label = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
    present = targets.sum(axis=0) > 0
    return {"micro_f1": float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
            "macro_f1_all": float(per_label.mean()),
            "macro_f1_present": float(per_label[present].mean()),
            "precision": float(tp.sum() / max(tp.sum() + fp.sum(), 1)),
            "recall": float(tp.sum() / max(tp.sum() + fn.sum(), 1))}

def multiclass_scores(probabilities: np.ndarray, targets: np.ndarray) -> dict[str, float]:
    predicted = probabilities.argmax(axis=1)
    seen = sorted(set(predicted) | set(targets))
    one_hot = np.eye(probabilities.shape[1])[:, seen]
    return {"accuracy": float((predicted == targets).mean()),
            "macro_f1": f1_scores(one_hot[predicted], one_hot[targets])["macro_f1_all"]}

def tune_threshold(probabilities: np.ndarray, targets: np.ndarray) -> float:
    return float(max(THRESHOLD_GRID, key=lambda threshold: f1_scores(probabilities >= threshold, targets)["micro_f1"]))

ITERATION1_LABELS = sorted({label for row in df.itertuples() for label in [*row.traits, *row.traditions]})

def official_label_matrix(frame: pd.DataFrame) -> np.ndarray:
    return np.array([[label in {*row.traits, *row.traditions} for label in ITERATION1_LABELS] for row in frame.itertuples()], dtype=np.int64)

def to_official(frame: pd.DataFrame, predicted_traits: np.ndarray, predicted_traditions: np.ndarray,
                columns: list[str] = FULL_INPUT) -> np.ndarray:
    output = np.zeros((len(frame), len(ITERATION1_LABELS)), dtype=np.int64)
    for row_index, row in enumerate(frame.itertuples()):
        labels = {trait_labels[i] for i in np.flatnonzero(predicted_traits[row_index])}
        labels |= {TRADITIONS[i] for i in np.flatnonzero(predicted_traditions[row_index])}
        if "category" in columns:  # the rule needs the category and cantrip input
            labels |= set(rule_traits(row.category, row.is_cantrip))
        for label in labels:
            output[row_index, ITERATION1_LABELS.index(label)] = 1
    return output

def evaluate(model: SpellTaggerV15, use_naive_bayes: bool, columns: list[str] = FULL_INPUT) -> tuple[dict[str, Any], dict[str, float]]:
    probabilities = {split: predict(model, build_inputs(split, use_naive_bayes, columns)) for split in ["valid", "test"]}
    thresholds = {group: tune_threshold(probabilities["valid"][group], Y["valid"][group]) for group in MULTILABEL_GROUPS}
    predicted = {group: probabilities["test"][group] >= thresholds[group] for group in MULTILABEL_GROUPS}
    report: dict[str, Any] = {group: f1_scores(predicted[group], Y["test"][group]) for group in MULTILABEL_GROUPS}
    report.update({group: multiclass_scores(probabilities["test"][group], Y["test"][group]) for group in MULTICLASS_GROUPS})
    official = to_official(test_df, predicted["traits"], predicted["traditions"], columns)
    report["comparable_100_labels"] = f1_scores(official, official_label_matrix(test_df))
    without_rules = [index for index, label in enumerate(ITERATION1_LABELS) if label not in RULE_TRAITS]
    report["comparable_without_rule_traits"] = f1_scores(official[:, without_rules], official_label_matrix(test_df)[:, without_rules])
    report = {group: {key: round(value, 4) for key, value in scores.items()} for group, scores in report.items()}
    return report, thresholds

def print_report(title: str, report: dict[str, Any]) -> None:
    print(f"\n  {title}")
    for group, scores in report.items():
        print(f"    {group:31s} " + ", ".join(f"{key}={value:.3f}" for key, value in scores.items()))

print("\n[4/5] Training and evaluating")
print("\n  Variant A: fixed BiGRU without Naive Bayes")
model_without_nb, history_without_nb = train(use_naive_bayes=False)
report_without_nb, thresholds_without_nb = evaluate(model_without_nb, use_naive_bayes=False)
print_report("Variant A (fixed BiGRU), test set", report_without_nb)

print("\n  Variant B: fixed BiGRU + Naive Bayes log-odds")
model, history = train(use_naive_bayes=True)
report, thresholds = evaluate(model, use_naive_bayes=True)
print(f"  tuned thresholds: {thresholds}")
print_report("Variant B (fixed BiGRU + Naive Bayes), test set", report)


[4/5] Training and evaluating

  Variant A: fixed BiGRU without Naive Bayes


  epoch=01 train_loss=4.3934 valid_loss=3.8149 * best


  epoch=02 train_loss=3.4442 valid_loss=3.4618 * best


  epoch=03 train_loss=3.0107 valid_loss=3.0563 * best


  epoch=04 train_loss=2.5930 valid_loss=2.7906 * best


...

  epoch=07 train_loss=1.5978 valid_loss=2.6552


  epoch=08 train_loss=1.4176 valid_loss=2.8012


  epoch=09 train_loss=1.2428 valid_loss=2.8799


  epoch=10 train_loss=1.1558 valid_loss=3.0071
  early stopping: no improvement for 5 epochs


  tuned thresholds: {'traits': 0.6, 'traditions': 0.35, 'damage_types': 0.7, 'sustained': 0.4}

  Variant B (fixed BiGRU + Naive Bayes), test set
    traits                          micro_f1=0.739, macro_f1_all=0.224, macro_f1_present=0.254, precision=0.837, recall=0.662
    traditions                      micro_f1=0.749, macro_f1_all=0.735, macro_f1_present=0.735, precision=0.633, recall=0.916
    damage_types                    micro_f1=0.217, macro_f1_all=0.115, macro_f1_present=0.115, precision=0.429, recall=0.145
    sustained                       micro_f1=0.510, macro_f1_all=0.510, macro_f1_present=0.510, precision=0.500, recall=0.520
    save                            accuracy=0.875, macro_f1=0.829
    area                            accuracy=0.823, macro_f1=0.348
    comparable_100_labels           micro_f1=0.740, macro_f1_all=0.197, macro_f1_present=0.252, precision=0.759, recall=0.721
    comparable_without_rule_traits  micro_f1=0.722, macro_f1_all=0.180, macro_f1_present=0

In [7]:
iteration1_metrics = json.loads((ROOT.parent / "iteration_1" / "artifacts" / "metrics.json").read_text(encoding="utf-8"))["traits"]
summary = pd.DataFrame([
    {"model": "iteration 1 - BiGRU", "traits+traditions micro_f1 (100 labels)": iteration1_metrics["micro_f1"],
     "macro_f1_all": iteration1_metrics["macro_f1"]},
    {"model": "1.5 A - fixed BiGRU", "traits+traditions micro_f1 (100 labels)": report_without_nb["comparable_100_labels"]["micro_f1"],
     "macro_f1_all": report_without_nb["comparable_100_labels"]["macro_f1_all"]},
    {"model": "1.5 B - fixed BiGRU + Naive Bayes", "traits+traditions micro_f1 (100 labels)": report["comparable_100_labels"]["micro_f1"],
     "macro_f1_all": report["comparable_100_labels"]["macro_f1_all"]},
]).round(3)
print("\n[5/5] Summary on the same 192 test spells")
print(summary.to_string(index=False))


[5/5] Summary on the same 192 test spells
                            model  traits+traditions micro_f1 (100 labels)  macro_f1_all
              iteration 1 - BiGRU                                    0.388         0.156
              1.5 A - fixed BiGRU                                    0.700         0.107
1.5 B - fixed BiGRU + Naive Bayes                                    0.740         0.197


## Test: the original idea – only rank + description

The original idea of the project: **the user types only the rank and the description of a spell, and the model fills in everything else.**

The best variant (B, with Naive Bayes) is trained again with only rank + description as input: no category, no cantrip, no casting time. Data, split, settings and test spells stay the same.

In iteration 1.5 the traits `cantrip` and `focus` come from a rule that uses the category and cantrip input. Without these inputs the rule cannot be used, so `cantrip` and `focus` are not predicted and count as misses in the main score. (Learning them instead does not work for this network: it does not learn `cantrip` at all, F1 = 0.)

In [8]:
print("  Variant B with only rank + description")
original_model, original_history = train(use_naive_bayes=True, columns=ORIGINAL_IDEA)
original_report, original_thresholds = evaluate(original_model, use_naive_bayes=True, columns=ORIGINAL_IDEA)

def main_scores(report: dict[str, Any]) -> dict[str, float]:
    return {"traits + traditions, 100 labels (micro-F1)": report["comparable_100_labels"]["micro_f1"],
            "traits (micro-F1)": report["traits"]["micro_f1"], "traditions (micro-F1)": report["traditions"]["micro_f1"],
            "save (accuracy)": report["save"]["accuracy"], "area (accuracy)": report["area"]["accuracy"],
            "damage types (micro-F1)": report["damage_types"]["micro_f1"], "sustained (F1)": report["sustained"]["micro_f1"]}

pd.DataFrame({"full input": main_scores(report), "only rank + description": main_scores(original_report)}).round(3)

  Variant B with only rank + description


  epoch=01 train_loss=3.3918 valid_loss=2.8605 * best


  epoch=02 train_loss=2.7616 valid_loss=2.6961 * best


  epoch=03 train_loss=2.5679 valid_loss=2.5706 * best


  epoch=04 train_loss=2.4005 valid_loss=2.5283 * best


...

  epoch=06 train_loss=1.9875 valid_loss=2.6420


  epoch=07 train_loss=1.7829 valid_loss=2.7655


  epoch=08 train_loss=1.5714 valid_loss=3.0111


  epoch=09 train_loss=1.4509 valid_loss=3.0170
  early stopping: no improvement for 5 epochs


,full input,only rank + description
"traits + traditions, 100 labels (micro-F1)",0.740,0.647
traits (micro-F1),0.739,0.734
traditions (micro-F1),0.749,0.620
save (accuracy),0.875,0.849
area (accuracy),0.823,0.833
damage types (micro-F1),0.217,0.280
sustained (F1),0.510,0.431


### What the model makes of rank + description

`predict_spell(rank, description)` is the original idea as a function. Two homebrew spells and three spells from the test set (with the correct Foundry answer below them). The same examples are used in every iteration.

In [9]:
HOMEBREW = [
    ("Lava Geyser (homebrew, project proposal)", 4,
     "You channel magical energy to rupture the earth, summoning a burst of molten rock. The eruption deals massive heat damage "
     "to anyone in the area and requires you to perform complex hand gestures to sustain it. Each creature in the area must "
     "attempt a basic Reflex save."),
    ("Molten Burst (homebrew, also used in iteration 1)", 3,
     "A burst of molten fire fills a 30-foot area. Creatures attempt a Reflex save and take persistent fire damage."),
]
REAL_SPELLS = ["Gust of Wind", "Suggestion", "Wordsmith"]  # from the test set; Wordsmith is a focus spell
FIELDS = ["traits", "traditions", "save", "area", "damage_types", "sustained"]

@torch.no_grad()
def predict_spell(rank: int, description: str) -> dict[str, list[str]]:
    row = pd.DataFrame([{"tokens": tokenize(clean_text(description)), "rank": rank}])
    ids = encode_tokens(row["tokens"][0])
    extra = np.hstack([structured_matrix(row, ORIGINAL_IDEA), (nb_log_odds(presence_matrix(row), nb_ratio, nb_prior) - nb_mean) / nb_std])
    original_model.eval()
    outputs = original_model(torch.tensor([ids]), torch.tensor([len(ids)]), torch.tensor(extra, dtype=torch.float32))
    probabilities = {group: (logits.sigmoid() if group in MULTILABEL_GROUPS else logits.softmax(dim=1)).numpy()[0]
                     for group, logits in outputs.items()}
    spell = {group: [label for label, probability in zip(MULTILABEL_GROUPS[group], probabilities[group]) if probability >= original_thresholds[group]]
             for group in MULTILABEL_GROUPS}
    spell.update({group: [MULTICLASS_GROUPS[group][int(probabilities[group].argmax())]] for group in MULTICLASS_GROUPS})
    return spell

def correct_answer(row: Any) -> dict[str, list[str]]:
    return {"traits": [label for label in row.traits if label in trait_labels or label in RULE_TRAITS], "traditions": row.traditions,
            "save": [row.save], "area": [row.area], "damage_types": [label for label in row.damage_types if label in damage_labels],
            "sustained": ["sustained"] if row.sustained else []}

def as_text(spell: dict[str, list[str]]) -> dict[str, str]:
    return {field: ", ".join(spell[field]) or "–" for field in FIELDS}

rows = [{"spell": name, "rank": rank, **as_text(predict_spell(rank, description))} for name, rank, description in HOMEBREW]
for name in REAL_SPELLS:
    spell = next(test_df[test_df["name"] == name].itertuples())
    rows.append({"spell": f"{name} (test set)", "rank": spell.rank, **as_text(predict_spell(spell.rank, spell.text))})
    rows.append({"spell": "↳ correct answer", "rank": spell.rank, **as_text(correct_answer(spell))})
pd.DataFrame(rows).set_index("spell")

,rank,traits,traditions,save,area,damage_types,sustained
spell,,,,,,,
"Lava Geyser (homebrew, project proposal)",4,"concentrate, manipulate","arcane, divine, primal",none,burst,–,–
"Molten Burst (homebrew, also used in iteration 1)",3,"concentrate, manipulate","arcane, divine, occult, primal",none,burst,–,–
Gust of Wind (test set),1,"concentrate, manipulate","arcane, primal",fortitude,burst,bludgeoning,–
↳ correct answer,1,"air, concentrate, manipulate","arcane, primal",fortitude,line,bludgeoning,–
Suggestion (test set),4,"concentrate, incapacitation, manipulate, mental","arcane, occult",will,none,–,–
↳ correct answer,4,"concentrate, incapacitation, linguistic, manip...","arcane, occult",will,none,–,–
Wordsmith (test set),4,"concentrate, manipulate","arcane, divine, occult",none,none,–,–
↳ correct answer,4,"concentrate, focus, manipulate",–,none,none,–,–


## Saved model

In [10]:
torch.save(model.state_dict(), ARTIFACT_DIR / "model.pt")
np.savez_compressed(ARTIFACT_DIR / "naive_bayes.npz", ratio=nb_ratio, prior=nb_prior, mean=nb_mean, std=nb_std)
metadata = {"token_to_id": token_to_id, "ngrams": feature_names, "trait_labels": trait_labels, "damage_labels": damage_labels,
            "multiclass_groups": MULTICLASS_GROUPS, "structured_values": structured_values, "thresholds": thresholds,
            "max_text_length": MAX_TEXT_LENGTH, "seed": SEED, "model_version": "bigru-naive-bayes-v1.5"}
(ARTIFACT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps({"without_naive_bayes": report_without_nb, "with_naive_bayes": report,
                                                       "only_rank_and_description": original_report}, indent=2), encoding="utf-8")
print(f"saved: {ARTIFACT_DIR / 'model.pt'}, naive_bayes.npz, metadata.json, metrics.json")

saved: /Users/bohdanlaskavyi/Fontys/year2september3/Personall challenge/PF2e-splells-creations/iterations/iteration_1_5/artifacts/model.pt, naive_bayes.npz, metadata.json, metrics.json


## Conclusions

**The fixes matter more than the model.** With the same model family as iteration 1 (embeddings learned from scratch + BiGRU), only the fixes from iteration 2 lift the score in the 100-label space from 0.388 (0.448 with a tuned threshold) to **0.700** (variant A). The fixes are: cleaning, structured input, proper targets, packed sequences, early stopping and tuned thresholds.

**Naive Bayes helps, but only through the shortcut.** With the Naive Bayes guess added to every output, the score rises to **0.740** (variant B). Trait micro-F1 goes from 0.672 to 0.739 and trait macro-F1 from 0.10 to 0.25, so rare traits benefit most. In a first attempt the Naive Bayes scores were only an extra input to the hidden layer. The GRU then overfitted within five epochs and mostly ignored them (0.703).

**What is still weak:**

* The GRU still learns its word vectors from 1,536 spells. It overfits after about 5 epochs, and early stopping keeps the best epoch.
* Damage types (micro-F1 0.22) and sustained (F1 0.51) are poor. One Naive Bayes number per label is too coarse for them. The NB-LR model of iteration 2 weighs every word separately and reaches 0.71 for damage types.
* In the examples of the test, `fire` (Molten Burst) and `air` (Gust of Wind) are missed. Iteration 2 gets both right.

**What is better:** the best save-type accuracy of all models (0.875), because reading word order helps with phrases like "attempt a basic Reflex save".

**Position in the project:** iteration 1.5 shows what the iteration 1 approach can reach once the data and evaluation are fixed: 0.740, between iteration 1 (0.388) and iteration 2 (0.781). Iteration 2 remains better with a simpler model.

**Test: only rank + description → 0.647** (full input: 0.740).

* Save, area and damage types stay about the same, because they are written in the description.
* The loss comes from what is not in the text: `cantrip` and `focus` (here they come from a rule that needs the category input) and the traditions of focus spells.
* In the examples the network predicts almost only `concentrate, manipulate`: with little data it falls back to the most common traits.